# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR^2 dataset package using the `mlcroissant` library. We will review the dataset structure, extract records from available record sets, and perform basic exploratory analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

_https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json_


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print("Description:")
print(metadata.description)


## 2. Data Overview
Review available record sets, fields, and their IDs.

We start by listing all record sets with their `@id`, then for each record set, list the available fields and columns by `@id`.

In [ ]:
# Explore record sets in the dataset
record_sets = list(dataset.record_sets())  # Returns list of RecordSet objects
print(f"Found {len(record_sets)} record sets:")

record_set_ids = []
for rs in record_sets:
    print(f"\nRecordSet Name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)

    # List fields (columns) in the RecordSet
    print("Fields/Columns:")
    for field in rs.fields:
        print(f"  - Name: {field.name} | @id: {field.id} | Type: {field.data_type}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All RecordSets and fields/columns are referenced by their `@id`.


In [ ]:
# Extract records from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for RecordSet @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by field values, normalizing numeric fields, and grouping data.

We will select a numeric field for demonstration, using its `@id`.

In [ ]:
# Example EDA: Identify numeric fields in the first record set
first_record_set_id = record_set_ids[0]
first_rs = [rs for rs in record_sets if rs.id == first_record_set_id][0]

numeric_field_ids = [f.id for f in first_rs.fields if f.data_type in ('Float', 'Integer', 'Number')]

# If numeric fields exist, proceed with filtering and normalization
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]
    print(f"Selected numeric field: {numeric_field_id}")

    df = dataframes[first_record_set_id]
    if numeric_field_id in df.columns:
        # Apply filter
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field
        group_field_ids = [f.id for f in first_rs.fields if f.data_type == 'Text']
        if group_field_ids:
            group_field_id = group_field_ids[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(grouped_df.head())
else:
    print("No numeric fields found in this RecordSet for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: Distribution of a numeric variable (`@id`) for the first RecordSet.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If numeric field is available, plot its histogram
if numeric_field_ids and numeric_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_ids and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR^2 dataset using the `mlcroissant` library. By referencing all entities via their `@id`, we demonstrated dataset access and basic EDA. The structure supports further clinical analysis, such as stratifying records, normalizing numeric clinical variables, and visualizing group differences.

Consider exploring additional record sets or fields and applying more advanced statistical or machine learning methods as needed.